# Exploratory Data Analysis — Amazon Books Reviews 2023

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})
BLUE, ORANGE, GREEN, RED = '#4C72B0', '#DD8452', '#55A868', '#C44E52'

RAW  = Path('../data/raw')
PROC = Path('../data/processed')

## Dataset Overview

In [ ]:
reviews  = pd.read_parquet(RAW / 'reviews_raw.parquet')
metadata = pd.read_parquet(RAW / 'metadata_raw.parquet')

print('=== Reviews ===')
print(f'  Rows          : {len(reviews):,}')
print(f'  Unique users  : {reviews["user_id"].nunique():,}')
print(f'  Unique books  : {reviews["asin"].nunique():,}')
print(f'  Date range    : {pd.to_datetime(reviews["timestamp"], unit="s").dt.year.min()} – {pd.to_datetime(reviews["timestamp"], unit="s").dt.year.max()}')
print(f'  Nulls         : {reviews.isnull().sum().to_dict()}')
print()
print('=== Metadata ===')
print(f'  Rows          : {len(metadata):,}')
print(f'  Columns       : {metadata.columns.tolist()}')
display(reviews.head(3))
display(metadata.head(3))

In [ ]:
n_users  = reviews['user_id'].nunique()
n_items  = reviews['asin'].nunique()
sparsity = 1 - len(reviews) / (n_users * n_items)
print(f'Matrix size : {n_users:,} × {n_items:,} = {n_users * n_items:,.0f} possible interactions')
print(f'Observed    : {len(reviews):,}   ({1 - sparsity:.4%} dense)')
print(f'Sparsity    : {sparsity:.4%}')

## Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = reviews['rating'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=BLUE, edgecolor='white', width=0.6)
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Raw Counts')
for x, y in zip(counts.index, counts.values):
    axes[0].text(x, y + counts.max() * 0.01, f'{y/len(reviews):.1%}',
                 ha='center', fontsize=9)

cum = counts.sort_index(ascending=False).cumsum() / len(reviews)
axes[1].plot(cum.index, cum.values * 100, marker='o', color=ORANGE)
axes[1].set_xlabel('Star Rating (from highest)')
axes[1].set_ylabel('Cumulative %')
axes[1].set_title('Cumulative Share (top-down)')
axes[1].invert_xaxis()
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Rating Distribution', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/01_rating_distribution.png', bbox_inches='tight')
plt.show()

print(f'Mean   : {reviews["rating"].mean():.3f}')
print(f'Median : {reviews["rating"].median():.1f}')
print(f'Skew   : {reviews["rating"].skew():.3f}')

## User Activity

In [ ]:
rpu = reviews.groupby('user_id').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rpu, bins=50, color=BLUE, edgecolor='white', log=True)
axes[0].set_xlabel('Ratings per User')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Activity Distribution')

x = np.sort(rpu.values)[::-1]
axes[1].loglog(np.arange(1, len(x)+1), x, '.', ms=2, color=BLUE, alpha=0.5)
axes[1].set_xlabel('User rank (log)')
axes[1].set_ylabel('Ratings (log)')
axes[1].set_title('Log–Log Plot')

sorted_vals = np.sort(rpu.values)
cum_share_users   = np.linspace(0, 1, len(sorted_vals))
cum_share_ratings = np.cumsum(sorted_vals) / sorted_vals.sum()
axes[2].plot(cum_share_users * 100, cum_share_ratings * 100, color=BLUE)
axes[2].plot([0, 100], [0, 100], '--', color='grey', label='Perfect equality')
axes[2].fill_between(cum_share_users * 100, cum_share_ratings * 100,
                     cum_share_users * 100, alpha=0.15, color=BLUE)
axes[2].set_xlabel('% of Users (least to most active)')
axes[2].set_ylabel('% of Ratings')
axes[2].set_title('Lorenz Curve')
axes[2].legend()

plt.suptitle('User Activity', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/02_user_activity.png', bbox_inches='tight')
plt.show()

print(rpu.describe().to_string())
top10_pct = rpu.head(int(len(rpu) * 0.1)).sum() / rpu.sum()
print(f'\nTop 10% of users account for {top10_pct:.1%} of all ratings')

## Item Popularity

In [ ]:
rpb = reviews.groupby('asin').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rpb, bins=60, color=GREEN, edgecolor='white', log=True)
axes[0].set_xlabel('Ratings per Book')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Popularity Distribution')

cumulative = rpb.cumsum() / rpb.sum()
axes[1].plot(np.arange(1, len(cumulative)+1), cumulative.values * 100, color=GREEN)
axes[1].axvline(x=int(len(rpb) * 0.2), color=RED, linestyle='--', label='Top 20% of books')
idx_20 = int(len(rpb) * 0.2)
axes[1].annotate(f'{cumulative.iloc[idx_20]:.0%} of ratings',
                 xy=(idx_20, cumulative.iloc[idx_20] * 100),
                 xytext=(idx_20 + len(rpb)*0.05, cumulative.iloc[idx_20] * 100 - 10),
                 arrowprops=dict(arrowstyle='->', color=RED), color=RED)
axes[1].set_xlabel('Books ranked by popularity')
axes[1].set_ylabel('Cumulative % of ratings')
axes[1].set_title('Cumulative Ratings')
axes[1].legend()

plt.suptitle('Item Popularity', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/03_item_popularity.png', bbox_inches='tight')
plt.show()

print(rpb.describe().to_string())
top20_items = rpb.head(int(len(rpb) * 0.20)).sum() / rpb.sum()
print(f'\nTop 20% of books account for {top20_items:.1%} of all ratings')

## Temporal Analysis

In [ ]:
reviews['date']       = pd.to_datetime(reviews['timestamp'], unit='s')
reviews['year_month'] = reviews['date'].dt.to_period('M')
reviews['year']       = reviews['date'].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

monthly = reviews.groupby('year_month').size()
monthly.index = monthly.index.to_timestamp()
axes[0].fill_between(monthly.index, monthly.values, alpha=0.4, color=BLUE)
axes[0].plot(monthly.index, monthly.values, color=BLUE, lw=1.5)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Reviews per Month')
axes[0].set_title('Review Volume')

yearly_mean = reviews.groupby('year')['rating'].mean()
axes[1].plot(yearly_mean.index, yearly_mean.values, marker='o', color=ORANGE)
axes[1].axhline(reviews['rating'].mean(), color='grey', linestyle='--', label='Overall mean')
axes[1].set_ylim(1, 5)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Mean Rating')
axes[1].set_title('Mean Rating by Year')
axes[1].legend()

plt.suptitle('Temporal Patterns', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/04_temporal.png', bbox_inches='tight')
plt.show()

## Genre Distribution

In [ ]:
genre_map       = metadata.set_index('asin')['genre'].to_dict()
reviews['genre'] = reviews['asin'].map(genre_map).fillna('Unknown')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_genres = reviews['genre'].value_counts().head(15)
axes[0].barh(top_genres.index[::-1], top_genres.values[::-1], color=BLUE)
axes[0].set_xlabel('Number of Reviews')
axes[0].set_title('Top 15 Genres by Review Volume')

genre_rating = (
    reviews[reviews['genre'].isin(top_genres.index)]
    .groupby('genre')['rating']
    .agg(['mean', 'std', 'count'])
    .sort_values('mean', ascending=True)
)
axes[1].barh(genre_rating.index, genre_rating['mean'], color=GREEN, xerr=genre_rating['std'],
             capsize=3, error_kw={'elinewidth': 1})
axes[1].axvline(reviews['rating'].mean(), color='grey', linestyle='--', label='Overall mean')
axes[1].set_xlabel('Mean Rating ± Std')
axes[1].set_title('Mean Rating by Genre')
axes[1].set_xlim(1, 5)
axes[1].legend()

plt.suptitle('Genre Landscape', fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/05_genre_analysis.png', bbox_inches='tight')
plt.show()

## Metadata Coverage

In [ ]:
coverage = {
    'Title':       (metadata['title'].str.len() > 1).mean(),
    'Description': (metadata['description'].str.len() > 50).mean(),
    'Genre':       (metadata['genre'].str.len() > 1).mean(),
    'Price':        metadata['price'].notna().mean(),
    'Cover Image': (metadata['cover_url'].str.len() > 5).mean(),
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    list(coverage.keys()),
    [v * 100 for v in coverage.values()],
    color=[GREEN if v > 0.7 else ORANGE if v > 0.4 else RED for v in coverage.values()],
)
for bar, val in zip(bars, coverage.values()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=10)
ax.set_xlim(0, 110)
ax.set_xlabel('Coverage %')
ax.set_title('Metadata Field Coverage')
plt.tight_layout()
plt.savefig('figures/06_metadata_coverage.png', bbox_inches='tight')
plt.show()

for field, val in coverage.items():
    print(f'{field:15s}: {val:.1%}')

## Sparsity & k-core Effect

In [ ]:
try:
    ratings = pd.read_parquet(PROC / 'ratings.parquet')
    train   = pd.read_parquet(PROC / 'train.parquet')
    val     = pd.read_parquet(PROC / 'val.parquet')
    test    = pd.read_parquet(PROC / 'test.parquet')
    processed_available = True
except FileNotFoundError:
    print('Processed data not yet available — run src/data/clean.py first.')
    processed_available = False

if processed_available:
    n_u      = ratings['user_idx'].nunique()
    n_i      = ratings['item_idx'].nunique()
    sparsity = 1 - len(ratings) / (n_u * n_i)

    rows = [
        {'Split': 'Raw',      'Users': reviews['user_id'].nunique(),  'Items': reviews['asin'].nunique(),        'Interactions': len(reviews),  'Sparsity': f"{1 - len(reviews)/(reviews['user_id'].nunique()*reviews['asin'].nunique()):.4%}"},
        {'Split': 'Filtered', 'Users': n_u, 'Items': n_i, 'Interactions': len(ratings), 'Sparsity': f'{sparsity:.4%}'},
        {'Split': 'Train',    'Users': train['user_idx'].nunique(), 'Items': train['item_idx'].nunique(), 'Interactions': len(train), 'Sparsity': '—'},
        {'Split': 'Val',      'Users': val['user_idx'].nunique(),   'Items': val['item_idx'].nunique(),   'Interactions': len(val),   'Sparsity': '—'},
        {'Split': 'Test',     'Users': test['user_idx'].nunique(),  'Items': test['item_idx'].nunique(),  'Interactions': len(test),  'Sparsity': '—'},
    ]
    display(pd.DataFrame(rows).set_index('Split'))